# Análise do Uso de Contexto pelos Arquivos AQG

Este notebook explora um problema associado ao formato AQG não ser ideal para uso direto como contexto de um projeto arquitetônico em sistemas de agentes baseados em LLMs. A hipótese inicial é a utilização do AQG apenas como fonte de verdade entre ambiente externo (formatos que clientes/arquitetos utilizam para construirem seus projetos) e ambiente interno (produtos da ArqGen). Assim, o AQG passa a ser utilizado como fonte para construção de memórias e estruturas para o agente operar.

O foco é responder a três perguntas:

1. Quanto cada AQG ocupa da janela de 1 milhão de tokens (contexto do modelo avaliado)?
2. Como a contagem muda entre o arquivo bruto, o JSON compacto e um recorte semântico?
3. Quanto o resumo usado pela aplicação ocupa em comparação com essas representações?

Todas as contagens vêm do endpoint do modelo, sem estimativas locais por caracteres ou linhas.

## Exemplo de Leitura do AQG

```text
Project
├── version
├── client_id, created_by, created_at, seed
├── site
│   ├── version, created_at, updated_at
│   ├── base_position {x, y, z, angle}
│   ├── inputs[]
│   ├── processed_info[]
│   ├── analysis
│   │   ├── avg_inclination
│   │   ├── inclination_groups
│   │   └── region_groups
│   └── elements[]
│       ├── hid, label, base_level_id, base_offset, height?
│       └── cad
│           ├── type, layer, geometry
│           └── faces
├── families[]
│   ├── class_path, label, description, tags[]
│   ├── properties
│   │   └── type
│   ├── volume[]
│   │   └── class_path, geometry, height, tags[]
│   ├── drawing_elements[]
│   │   └── class_path, geometry, label, tags[]
│   ├── blocks[]
│   └── metadados de identidade, autoria e datas
├── solution
│   ├── placed_objects[]
│   │   ├── PlacedFamily
│   │   │   └── hid, index, base_level_id, position {x, y, z, angle}
│   │   ├── PlacedElement
│   │   │   └── hid, index, cad, atributos de implantação
│   │   └── PlacedAuxiliaryArea
│   │       └── hid, index, cad, atributos de área
│   ├── outputs[]
│   ├── scores {}
│   ├── annotations[]
│   ├── strategy[]
│   └── road_network
└── archprog[]
```

## Fluxo Atual da Aplicação

```text
AQG -> load_aqg() -> CadModel em memória
                        |
                        +-> describe_model() -> resumo sem números no system prompt --+
                        |                                                             |
                        +-> 16 tools -> JSON sob demanda -----------------------------+-> LLM
                        |                                                             |
                        +-> prims_payload() -> viewer, fora do contexto               |
                                                                                      |
instruções + schemas das tools + usuário + histórico ---------------------------------+
```

## 1. Setup

In [1]:
import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import anthropic
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "cad" / "model.py").exists():
            return candidate
    raise RuntimeError("Abra o notebook dentro do repositório do projeto.")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data/aqg"
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.api.core.config import get_app_settings, require_ai_gateway  # noqa: E402
from src.cad.model import describe_model, load_aqg  # noqa: E402

CONTEXT_WINDOW_TOKENS = 1_000_000


def compact(obj: Any) -> str:
    """Serializa o JSON sem indentação, preservando seus dados."""
    return json.dumps(obj, separators=(",", ":"), ensure_ascii=False)


def semantic_core(project: dict) -> dict:
    """Recorte para comparar tokens, sem validar relações ou indicadores."""
    site = project.get("site") or {}
    solution = project.get("solution") or {}
    return {
        "inputs": site.get("inputs"),
        "processed_info": site.get("processed_info"),
        "outputs": solution.get("outputs"),
        "scores": solution.get("scores"),
        "archprog": project.get("archprog"),
        "families": [
            {
                key: family.get(key)
                for key in ("class_path", "label", "type", "properties")
            }
            for family in project.get("families") or []
        ],
        "placements": [
            {key: obj.get(key) for key in ("hid", "index", "position")}
            for obj in solution.get("placed_objects") or []
            if obj.get("type") == "PlacedFamily"
        ],
    }

### Arquivos Avaliados

O acervo inteiro participa da medição, sem seleção por tamanho ou qualidade. Os arquivos são identificados pelo caminho relativo a `data/aqg/`. O hash SHA-256 dos bytes originais fica registrado em `tokens_df` para identificar o conteúdo medido.

In [2]:
PATHS = sorted(DATA_DIR.rglob("*.json"))
if not PATHS:
    raise FileNotFoundError(f"Nenhum AQG em {DATA_DIR} (ver data/aqg/README.md)")

RAW: dict[str, str] = {}
PROJECTS: dict[str, dict] = {}
FILE_HASHES: dict[str, str] = {}
for path in PATHS:
    name = str(path.relative_to(DATA_DIR))
    data = path.read_bytes()
    RAW[name] = data.decode("utf-8")
    PROJECTS[name] = json.loads(RAW[name])
    FILE_HASHES[name] = hashlib.sha256(data).hexdigest()

print(f"{len(PROJECTS)} AQGs; quatro representações por arquivo.")
print(f"{len(PROJECTS) * 4} contagens no endpoint do modelo.")
display(pd.Series([p.get("version") for p in PROJECTS.values()]).value_counts())

24 AQGs; quatro representações por arquivo.
96 contagens no endpoint do modelo.


18.0.0    15
17.1.0     2
7.0.2      1
12.6.0     1
12.4.1     1
12.7.0     1
16.4.0     1
16.8.0     1
17.7.0     1
Name: count, dtype: int64

## 2. Contagem no Endpoint do Modelo

Cada representação é enviada integralmente como uma única mensagem de usuário ao endpoint `/v1/messages/count_tokens`:

- **Bruto:** texto original do arquivo AQG.
- **Compacto:** o mesmo JSON serializado sem indentação.
- **Núcleo semântico:** recorte com parâmetros e dados processados do terreno, indicadores, scores, programa arquitetônico, propriedades das famílias e posicionamentos das instâncias. Exclui outras informações, como malhas, polígonos e `site.analysis`; não representa todo o significado do projeto.
- **Resumo atual:** texto produzido por `describe_model()` e usado pela aplicação para apresentar o documento ao agente.

O endpoint retorna uma estimativa de tokens para o modelo indicado (conforme a Anthropic declara). A referência é a janela de 1 milhão de tokens do Sonnet 4.6. Se o modelo configurado mudar, a referência deve ser conferida novamente. [Contagem de tokens](https://platform.claude.com/docs/en/build-with-claude/token-counting?q=count_tokens) e [janela do Sonnet 4.6](https://platform.claude.com/docs/en/models/sonnet-4-6/overview).

As contagens incluem a mensagem enviada, mas não as instruções, os schemas das ferramentas, o histórico nem a saída do agente. O resumo é medido isoladamente para comparação, embora na aplicação seja parte do system prompt. Compactar preserva os dados do JSON; núcleo e resumo descartam informações, portanto não são alternativas de conteúdo equivalente.

Executar a célula envia as quatro representações de todos os arquivos ao gateway configurado.

In [3]:
def count_tokens(client: anthropic.Anthropic, model_id: str, text: str) -> int:
    """Consulta o contador do modelo; não usa estimativas locais como fallback."""
    result = client.messages.count_tokens(
        model=model_id,
        messages=[{"role": "user", "content": text}],
    )
    return result.input_tokens


require_ai_gateway()
settings = get_app_settings()
MODEL_ID = settings.ANTHROPIC_MODEL
MEASUREMENT_STARTED_UTC = datetime.now(timezone.utc).isoformat()
TOKEN_ROWS = []
print("Modelo:", MODEL_ID)
print("Início (UTC):", MEASUREMENT_STARTED_UTC)

# O SDK acrescenta /v1/messages/count_tokens à raiz do gateway.
with anthropic.Anthropic(
    base_url=settings.PROXY_AI_BASE_URL.rstrip("/").removesuffix("/v1"),
    api_key=settings.LITELLM_API_KEY,
    timeout=180,
    max_retries=2,
) as client:
    for index, (name, project) in enumerate(PROJECTS.items(), start=1):
        print(f"[{index}/{len(PROJECTS)}] {name}", flush=True)
        cad_model = load_aqg(RAW[name].encode(), name)
        representations = {
            "bruto": RAW[name],
            "compacto": compact(project),
            "nucleo": compact(semantic_core(project)),
            "resumo_atual": describe_model(cad_model),
        }
        row = {
            "arquivo": name,
            "versao": project.get("version"),
            "sha256": FILE_HASHES[name],
            "modelo": MODEL_ID,
        }
        for variant, text in representations.items():
            row[f"tokens_{variant}"] = count_tokens(client, MODEL_ID, text)
        row["medido_em_utc"] = datetime.now(timezone.utc).isoformat()
        TOKEN_ROWS.append(row)

tokens_df = pd.DataFrame(TOKEN_ROWS).sort_values(
    "tokens_bruto", ascending=False, ignore_index=True
)
TOKEN_COLUMNS = [
    "tokens_bruto",
    "tokens_compacto",
    "tokens_nucleo",
    "tokens_resumo_atual",
]
display(tokens_df[["arquivo", "versao", *TOKEN_COLUMNS]])

Modelo: claude-sonnet-4-6
Início (UTC): 2026-10-30T13:58:05.405043+00:00
[1/24] avaliadas/01/bom_6ace404e4a7a894b01852f19.json
[2/24] avaliadas/01/otimo_6ace4108cfe0941cdf9a9189.json
[3/24] avaliadas/01/ruim_6ace40534a7a894b01852f1d.json
[4/24] avaliadas/02/bom_6ace3c8be748be1a355e6769.json
[5/24] avaliadas/02/otimo_6ace4079bf22db620be742d4.json
[6/24] avaliadas/02/ruim_6ace3c16d8664cf36a13dbb9.json
[7/24] avaliadas/03/bom_6ace7b93445b3a60fcb6162b.json
[8/24] avaliadas/03/otimo_6ace84f467fab4d8049d1394.json
[9/24] avaliadas/03/ruim_6ace7f0d246ca4a63fa193fd.json
[10/24] avaliadas/04/bom_6ace64863afd82bf6be93e16.json
[11/24] avaliadas/04/otimo_6ace64da8d16ae86ca3419e5.json
[12/24] avaliadas/04/ruim_6ace648d563a33090fea89d6.json
[13/24] avaliadas/05/bom_6ace4f3f9aee62f656aae2cc.json
[14/24] avaliadas/05/otimo_6ace52a0bc31d7e3e85c9509.json
[15/24] avaliadas/05/ruim_6ace4ea8ecd8aeaf968b312c.json
[16/24] favoritadas/6842c89845e64a9f238fba3c_6842c996181cf815481d1967.json
[17/24] favoritadas/6

,arquivo,versao,tokens_bruto,tokens_compacto,tokens_nucleo,tokens_resumo_atual
0,avaliadas/04/otimo_6ace64da8d16ae86ca3419e5.json,18.0.0,4549998,2813752,24661,964
1,avaliadas/01/otimo_6ace4108cfe0941cdf9a9189.json,18.0.0,4190155,2550003,6724,812
2,avaliadas/04/bom_6ace64863afd82bf6be93e16.json,18.0.0,2156625,1315552,11888,986
3,favoritadas/6abab05295f499e81abeca64_6abd57caa4c6bcd7c7e56699.json,17.7.0,1762996,1064503,13654,894
4,avaliadas/01/bom_6ace404e4a7a894b01852f19.json,18.0.0,1578741,943967,3361,785
5,avaliadas/05/otimo_6ace52a0bc31d7e3e85c9509.json,18.0.0,1440077,880333,23355,954
6,avaliadas/05/bom_6ace4f3f9aee62f656aae2cc.json,18.0.0,1122724,681730,17639,954
7,avaliadas/01/ruim_6ace40534a7a894b01852f1d.json,18.0.0,1081274,639524,3061,770
8,avaliadas/04/ruim_6ace648d563a33090fea89d6.json,18.0.0,953346,567922,5231,986
9,favoritadas/6842c89845e64a9f238fba3c_6842c996181cf815481d1967.json,7.0.2,818710,473085,7936,675


## 3. Ocupação da Janela

Uma proporção acima de 1 significa que a representação sozinha excede a janela de referência. Uma proporção menor ou igual a 1 não garante que a conversa caiba: ainda é necessário espaço para instruções, ferramentas, histórico e resposta.

In [4]:
window_df = tokens_df[["arquivo"]].copy()
summary_rows = []
for column in TOKEN_COLUMNS:
    variant = column.removeprefix("tokens_")
    counts = tokens_df[column]
    window_df[f"{variant}_x_janela_1m"] = counts / CONTEXT_WINDOW_TOKENS
    summary_rows.append(
        {
            "representacao": variant,
            "arquivos_medidos": len(counts),
            "tokens_min": counts.min(),
            "tokens_mediana": counts.median(),
            "tokens_max": counts.max(),
            "acima_de_1m": int((counts > CONTEXT_WINDOW_TOKENS).sum()),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
display(window_df)

,representacao,arquivos_medidos,tokens_min,tokens_mediana,tokens_max,acima_de_1m
0,bruto,24,26824,"675,208.50",4549998,8
1,compacto,24,15688,"403,373.50",2813752,4
2,nucleo,24,3061,"8,012.00",35705,0
3,resumo_atual,24,675,871.00,986,0


,arquivo,bruto_x_janela_1m,compacto_x_janela_1m,nucleo_x_janela_1m,resumo_atual_x_janela_1m
0,avaliadas/04/otimo_6ace64da8d16ae86ca3419e5.json,4.55,2.81,0.02,0.00
1,avaliadas/01/otimo_6ace4108cfe0941cdf9a9189.json,4.19,2.55,0.01,0.00
2,avaliadas/04/bom_6ace64863afd82bf6be93e16.json,2.16,1.32,0.01,0.00
3,favoritadas/6abab05295f499e81abeca64_6abd57caa4c6bcd7c7e56699.json,1.76,1.06,0.01,0.00
4,avaliadas/01/bom_6ace404e4a7a894b01852f19.json,1.58,0.94,0.00,0.00
5,avaliadas/05/otimo_6ace52a0bc31d7e3e85c9509.json,1.44,0.88,0.02,0.00
6,avaliadas/05/bom_6ace4f3f9aee62f656aae2cc.json,1.12,0.68,0.02,0.00
7,avaliadas/01/ruim_6ace40534a7a894b01852f1d.json,1.08,0.64,0.00,0.00
8,avaliadas/04/ruim_6ace648d563a33090fea89d6.json,0.95,0.57,0.01,0.00
9,favoritadas/6842c89845e64a9f238fba3c_6842c996181cf815481d1967.json,0.82,0.47,0.01,0.00


## 4. Redução de Tokens por Representação

A redução é calculada diretamente das contagens retornadas pelo endpoint para cada arquivo. `reducao_compacto_pct` compara o JSON bruto com o compacto; as outras colunas comparam o bruto com recortes que contêm menos informações. Esses fatores não medem economia total de uma conversa nem custo financeiro, visto que nossa preocupação é a gestão da janela.

In [5]:
reduction_df = tokens_df[["arquivo"]].copy()
for variant in ("compacto", "nucleo", "resumo_atual"):
    reduction_df[f"reducao_{variant}_pct"] = 100 * (
        1 - tokens_df[f"tokens_{variant}"] / tokens_df["tokens_bruto"]
    )
reduction_df["bruto_por_resumo_x"] = (
    tokens_df["tokens_bruto"] / tokens_df["tokens_resumo_atual"]
)
display(reduction_df)

,arquivo,reducao_compacto_pct,reducao_nucleo_pct,reducao_resumo_atual_pct,bruto_por_resumo_x
0,avaliadas/04/otimo_6ace64da8d16ae86ca3419e5.json,38.16,99.46,99.98,"4,719.91"
1,avaliadas/01/otimo_6ace4108cfe0941cdf9a9189.json,39.14,99.84,99.98,"5,160.29"
2,avaliadas/04/bom_6ace64863afd82bf6be93e16.json,39.00,99.45,99.95,"2,187.25"
3,favoritadas/6abab05295f499e81abeca64_6abd57caa4c6bcd7c7e56699.json,39.62,99.23,99.95,"1,972.03"
4,avaliadas/01/bom_6ace404e4a7a894b01852f19.json,40.21,99.79,99.95,"2,011.14"
5,avaliadas/05/otimo_6ace52a0bc31d7e3e85c9509.json,38.87,98.38,99.93,"1,509.51"
6,avaliadas/05/bom_6ace4f3f9aee62f656aae2cc.json,39.28,98.43,99.92,"1,176.86"
7,avaliadas/01/ruim_6ace40534a7a894b01852f1d.json,40.85,99.72,99.93,"1,404.25"
8,avaliadas/04/ruim_6ace648d563a33090fea89d6.json,40.43,99.45,99.90,966.88
9,favoritadas/6842c89845e64a9f238fba3c_6842c996181cf815481d1967.json,42.22,99.03,99.92,"1,212.90"


## 5. Resultados

Os resultados abaixo são calculados a partir de todos os arquivos medidos nesta execução.

In [6]:
def format_int(value: int) -> str:
    return f"{int(value):,}".replace(",", ".")


stats = summary_df.set_index("representacao")
n_files = len(tokens_df)
raw_exceeds = int(stats.loc["bruto", "acima_de_1m"])
compact_exceeds = int(stats.loc["compacto", "acima_de_1m"])
core_exceeds = int(stats.loc["nucleo", "acima_de_1m"])
summary_exceeds = int(stats.loc["resumo_atual", "acima_de_1m"])

display(
    Markdown(
        f"Foram medidos **{n_files} arquivos**, em quatro representações, "
        f"com o modelo **{MODEL_ID}**.\n\n"
        f"- **Bruto:** de {format_int(stats.loc['bruto', 'tokens_min'])} a "
        f"{format_int(stats.loc['bruto', 'tokens_max'])} tokens; "
        f"**{raw_exceeds}/{n_files}** excedem a janela de 1 milhão.\n"
        f"- **Compacto:** de {format_int(stats.loc['compacto', 'tokens_min'])} a "
        f"{format_int(stats.loc['compacto', 'tokens_max'])} tokens; "
        f"**{compact_exceeds}/{n_files}** excedem a janela.\n"
        f"- **Núcleo semântico:** de {format_int(stats.loc['nucleo', 'tokens_min'])} "
        f"a {format_int(stats.loc['nucleo', 'tokens_max'])} tokens; "
        f"**{core_exceeds}/{n_files}** excedem a janela.\n"
        f"- **Resumo atual:** de "
        f"{format_int(stats.loc['resumo_atual', 'tokens_min'])} a "
        f"{format_int(stats.loc['resumo_atual', 'tokens_max'])} tokens; "
        f"**{summary_exceeds}/{n_files}** excedem a janela."
    )
)

Foram medidos **24 arquivos**, em quatro representações, com o modelo **claude-sonnet-4-6**.

- **Bruto:** de 26.824 a 4.549.998 tokens; **8/24** excedem a janela de 1 milhão.
- **Compacto:** de 15.688 a 2.813.752 tokens; **4/24** excedem a janela.
- **Núcleo semântico:** de 3.061 a 35.705 tokens; **0/24** excedem a janela.
- **Resumo atual:** de 675 a 986 tokens; **0/24** excedem a janela.

O notebook mede o tamanho das representações, não a qualidade das respostas. Os resultados não demonstram se o agente consegue interpretar o núcleo semântico corretamente. Essa avaliação será realizada após a definição das estruturas de memória e comparação entre a solução atual e solução nova.

## 6. Proposta: Memória Intermediária

As medições mostram que a injeção integral do AQG não atende todo o acervo, mesmo após compactação, enquanto o núcleo semântico e o resumo ocupam uma parcela menor da janela. Isso motiva uma hipótese de arquitetura: manter o AQG como fonte dos dados e oferecer ao agente acesso seletivo a uma representação de domínio. O ganho dessa abordagem sobre a aplicação atual ainda precisa ser avaliado.

```text
C0  AQG bruto           fonte dos dados, comunicação entre externo -> ArqGen, versionado
 |
C1  modelo de domínio   objetos tipados + relações + fatos com unidade e origem
 |
C2  acesso do agente    mapa sem valores no prompt + consultas e cálculos sob demanda
 |
agente <-> C3 memória de trabalho
              fatos recuperados, hipóteses e decisões; persistência por projeto
              com referência às versões das soluções utilizadas
```

- **C0 - Fonte preservada.** O arquivo original permite conferir e reconstruir as demais representações. Nesta proposta, seu conteúdo integral não é enviado ao modelo; cada importação é identificada pelo projeto, pela solução e pela versão ou hash do arquivo.
- **C1 - Representação de domínio.** Normalizar o AQG uma vez em objetos como terreno, torre, família e indicador. Cada fato registra valor, unidade quando conhecida e caminho de origem; fatos calculados também registram a regra e os dados utilizados. Relações ambíguas e dados ausentes permanecem explícitos. A identidade deve distinguir projeto, solução e versão. Geometria volumosa pode permanecer fora dessa representação, com cálculos sob demanda.
- **C2 - Contexto sob demanda.** O prompt recebe um mapa do que pode ser consultado. Consultas filtradas retornam somente os fatos necessários, com procedência, e ferramentas executam os cálculos.
- **C3 - Continuidade do trabalho.** Persistir os fatos recuperados e o estado da tarefa, distinguindo-os das hipóteses do agente e das decisões do usuário. Comparações registram quais soluções e versões foram utilizadas. Quando o AQG muda, fatos armazenados que dependem dele precisam ser revalidados ou invalidados. A cada interação, apenas a parte relevante dessa memória retorna ao contexto.

A aplicação já implementa parte desse caminho com `CadModel`, `describe_model()` e ferramentas de consulta. A proposta é evoluir essa base com procedência explícita, organização de domínio e controle da memória de trabalho. Persistir informações, por si só, não reduz tokens: a redução depende de selecionar o que entra em cada requisição e evitar recuperar repetidamente dados desnecessários.

O próximo experimento deve comparar a aplicação atual e um protótipo dessa arquitetura nas mesmas perguntas, arquivos e modelo. Assim será possível verificar se a memória melhora o uso do contexto sem perder a qualidade das respostas.